## LLM Ideological Benchmark: Statistical Visualizations
**Project:** Geopolitical Mirror - US vs. Chinese LLM Bias Analysis  
**Models:** Llama-3.2-1B (US) vs. Qwen-2.5-1.5B (China)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set professional scientific style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.figsize': (10, 6),
    'figure.dpi': 100
})

# Color palette: Llama (Blue), Qwen (Red)
COLORS = {'Llama': '#4C72B0', 'Qwen': '#C44E52'}

In [ ]:
# Load data
df = pd.read_csv("../data/per_question_detail.csv")

# Use the Llama-vs-Qwen pairwise columns as the notebook's main gap/significance fields.
df['gap'] = df['gap_llama_qwen']
df['sig_95'] = df['sig95_llama_qwen']

# Calculate Refusal Rates (Total possible N per question is 30 based on 3 variants * 10 runs)
TOTAL_RESPONSES_PER_QUESTION = 30
df['llama_refusal_rate'] = (TOTAL_RESPONSES_PER_QUESTION - df['llama_n']) / TOTAL_RESPONSES_PER_QUESTION
df['qwen_refusal_rate'] = (TOTAL_RESPONSES_PER_QUESTION - df['qwen_n']) / TOTAL_RESPONSES_PER_QUESTION
print(f"Data loaded: {len(df)} questions analyzed.")

### Figure 1: Mean Likert Scores by Domain
This chart compares the average agreement scores (1-5) across ideological domains. 
*   **Interpretation:** Significant gaps in Moral and Scientific domains highlight cultural divergence.

In [ ]:
domain_means = df.groupby('domain')[['llama_mean', 'qwen_mean']].mean().reset_index()
domain_means['domain'] = domain_means['domain'].str.replace('/', '\n').str.title()

x = np.arange(len(domain_means))
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, domain_means['llama_mean'], width, label='Llama-3.2 (US)', color=COLORS['Llama'])
rects2 = ax.bar(x + width/2, domain_means['qwen_mean'], width, label='Qwen-2.5 (China)', color=COLORS['Qwen'])

ax.set_ylabel('Mean Likert Score (1-5)')
ax.set_title('Ideological Lean by Domain')
ax.set_xticks(x)
ax.set_xticklabels(domain_means['domain'])
ax.axhline(3.0, color='black', linestyle='--', alpha=0.5, label='Neutral')
ax.set_ylim(1, 5)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Figure 2: Refusal Rates by Domain
Shows which topics are most "sensitive" for each model.
*   **Interpretation:** High refusal rates indicate the presence of safety guardrails or ideological taboos.

In [ ]:
domain_refusal = df.groupby('domain')[['llama_refusal_rate', 'qwen_refusal_rate']].mean().reset_index()
domain_refusal['domain'] = domain_refusal['domain'].str.replace('/', '\n').str.title()

fig, ax = plt.subplots()
ax.bar(x - width/2, domain_refusal['llama_refusal_rate'] * 100, width, label='Llama-3.2', color=COLORS['Llama'], alpha=0.6)
ax.bar(x + width/2, domain_refusal['qwen_refusal_rate'] * 100, width, label='Qwen-2.5', color=COLORS['Qwen'], alpha=0.6)

ax.set_ylabel('Refusal Rate (%)')
ax.set_title('Model Sensitivity: Refusal Rate by Domain')
ax.set_xticks(x)
ax.set_xticklabels(domain_refusal['domain'])
ax.legend()

plt.tight_layout()
plt.show()

### Figure 3: Top 10 Ideological Flashpoints
Specific questions with the largest numeric divergence between the US and Chinese models.

In [ ]:
print(df.columns)

top_10 = df.sort_values('gap_llama_qwen', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = np.arange(len(top_10))

ax.barh(y_pos, top_10['gap_llama_qwen'], color='teal', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(top_10['question_id'] + " (" + top_10['domain'] + ")")
ax.invert_yaxis()  # labels read top-to-bottom
ax.set_xlabel('Numeric Gap (Absolute Difference)')
ax.set_title('Top 10 Most Polarized Questions (Flashpoints)')

plt.tight_layout()
plt.show()

### Figure 4: Response Distribution (Polarization vs. Neutrality)
A direct comparison of how often each score (1-5) was selected.
*   **Interpretation:** Qwen exhibits heavy clustering at '3' (Strategic Neutrality), while Llama shows a broader, more opinionated distribution.

In [ ]:
# Aggregated counts from the final analysis report
scores = [1, 2, 3, 4, 5]
llama_counts = [62, 144, 508, 188, 21]
qwen_counts = [1, 8, 932, 51, 45]

x_dist = np.arange(len(scores))
width_dist = 0.4

fig, ax = plt.subplots()
ax.bar(x_dist - width_dist/2, llama_counts, width_dist, label='Llama-3.2', color=COLORS['Llama'], alpha=0.8)
ax.bar(x_dist + width_dist/2, qwen_counts, width_dist, label='Qwen-2.5', color=COLORS['Qwen'], alpha=0.8)

ax.set_xlabel('Likert Score (1=Strong Agree, 5=Strong Disagree)')
ax.set_ylabel('Frequency')
ax.set_title('Frequency Distribution of Model Ratings')
ax.set_xticks(x_dist)
ax.set_xticklabels(scores)
ax.legend()

plt.tight_layout()
plt.show()

### Figure 5: Quadrant Analysis (Consistency vs. Divergence)
This scatter plot identifies the most robust data points.
*   **Robust Divergence (Bottom-Right):** High gap between models, high internal stability. These are the most credible indicators of bias.

In [ ]:
df['avg_instability'] = (df['llama_cross_var_nSD'] + df['qwen_cross_var_nSD']) / 2

plt.figure(figsize=(11, 7))
scatter = sns.scatterplot(data=df, x='gap_llama_qwen', y='avg_instability', 
                         hue='domain', palette='deep', s=120, edgecolors='w', alpha=0.8)

# Significance markers
sig_df = df[df['sig_95'] == True]
plt.scatter(sig_df['gap_llama_qwen'], sig_df['avg_instability'], s=150, facecolors='none', edgecolors='black', linewidth=1.5, label='Stat. Significant')

plt.axvline(x=0.5, color='black', linestyle='--', linewidth=1, alpha=0.3)
plt.axhline(y=0.15, color='black', linestyle='--', linewidth=1, alpha=0.3)

# Clean Background Shading (No floating text)
plt.fill_between([0.5, df['gap_llama_qwen'].max()*1.1], 0, 0.15, color='green', alpha=0.05, label='Robust Divergence')
plt.fill_between([0, 0.5], 0, 0.15, color='blue', alpha=0.05, label='Robust Consensus')
plt.fill_between([0.5, df['gap_llama_qwen'].max()*1.1], 0.15, df['avg_instability'].max()*1.1, color='orange', alpha=0.05, label='Noisy Bias')
plt.fill_between([0, 0.5], 0.15, df['avg_instability'].max()*1.1, color='red', alpha=0.05, label='High Uncertainty')

plt.title('The Consistency-Divergence Quadrant Analysis')
plt.xlabel('Numeric Gap (Absolute Difference)')
plt.ylabel('Within-Model Instability (Mean nSD)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.show()

### Figure 6: Uncertainty vs. Suggestibility
Comparing how much models flip-flop across runs (Uncertainty) vs. across phrasings (Suggestibility).

In [ ]:
stability_metrics = ['Framing Instability\n(Suggestibility)', 'Stochastic Instability\n(Uncertainty)']
llama_stab = [0.1096, 0.1736]
qwen_stab = [0.0477, 0.0818]

x_stab = np.arange(len(stability_metrics))

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(x_stab - width/2, llama_stab, width, label='Llama-3.2 (US)', color=COLORS['Llama'])
ax.bar(x_stab + width/2, qwen_stab, width, label='Qwen-2.5 (China)', color=COLORS['Qwen'])

ax.set_ylabel('Instability Metric (nSD)')
ax.set_title('Robustness Comparison: US vs. Chinese Models')
ax.set_xticks(x_stab)
ax.set_xticklabels(stability_metrics)
ax.legend()

plt.tight_layout()
plt.show()